# 02 — Layer-wise Activation Extraction from PIE

## Objective

This notebook implements a reproducible pipeline for
extracting internal activations from all Transformer
blocks of the pretrained PIE model.

The extracted representations will be used to train
separate Sparse Autoencoders (SAEs) for each layer.

## Research Questions

1. How can we extract activations from all PIE layers?
2. What constitutes an observation for SAE training?
3. How should latent and gene-specific representations
   be organized?
4. How can activations be stored efficiently?
5. How can we verify the integrity and reproducibility
   of the extracted datasets?

## Experimental Setup

We investigate three components:

- Encoder: 4 Transformer blocks.
- Processor: 6 Transformer blocks.
- Decoder: 4 Transformer blocks.

The hidden dimension is 768.

We use the official pretrained PIE checkpoint and
the original training and validation splits.

The pretrained PIE parameters remain frozen throughout
the activation extraction process.

## Methodological References

1. Verma et al. (2026).
   *PIE: Generalizing perturbation effects across
   unseen perturbations, contexts and datasets.*

2. Kendiukhov (2026).
   *Sparse autoencoders reveal organized biological
   knowledge but minimal regulatory logic in
   single-cell foundation models.*

## Scope

This notebook focuses on extracting, organizing,
and validating internal representations.

SAE training and feature interpretation are performed
in subsequent notebooks.

## 1. Environment and Project Configuration

The project uses an isolated Python environment and
the official PIE implementation included as a Git submodule.

All paths are resolved relative to the current project.

The pretrained checkpoint and biological data are loaded
from project-local directories.

No files or variables from previous research projects
are required.

The computational environment is inspected before
activation extraction.

In [1]:
import os
import sys
import time
import random

from pathlib import Path

import numpy as np
import torch


# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PIE_ROOT = PROJECT_ROOT / "external" / "pie"

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "models"
    / "replogle_wdataset"
    / "k562"
)

DATA_ROOT = PROJECT_ROOT / "data"
CACHE_ROOT = PROJECT_ROOT / "cache"
RUNS_ROOT = PROJECT_ROOT / "runs"

for path in [DATA_ROOT, CACHE_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# PIE environment variables
# --------------------------------------------------

os.environ["PIE_DATA_ROOT"] = str(DATA_ROOT)
os.environ["PIE_CACHE_DIR"] = str(CACHE_ROOT)
os.environ["PIE_RUNS_ROOT"] = str(RUNS_ROOT)


# --------------------------------------------------
# Official PIE source
# --------------------------------------------------

PIE_SRC = PIE_ROOT / "src"

if str(PIE_SRC) not in sys.path:
    sys.path.insert(0, str(PIE_SRC))


# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# --------------------------------------------------
# Verification
# --------------------------------------------------

assert PIE_ROOT.is_dir()
assert (CHECKPOINT_DIR / "best_auprc.ckpt").is_file()

print("Project:", PROJECT_ROOT)
print("PIE source:", PIE_ROOT)
print("Checkpoint:", CHECKPOINT_DIR)

print("\nEnvironment:")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nRandom seed:", SEED)

Project: D:\bio_paper\pie-sae
PIE source: D:\bio_paper\pie-sae\external\pie
Checkpoint: D:\bio_paper\pie-sae\models\replogle_wdataset\k562

Environment:
Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce GTX 1650

Random seed: 42


## 2. Loading the Pretrained PIE Model

We independently reconstruct the pretrained PIE model.

The procedure follows the official PIE implementation:

1. Load the pretrained checkpoint.
2. Initialize the official data module.
3. Prepare dataset-dependent metadata.
4. Reconstruct the model architecture.
5. Restore pretrained parameters.
6. Freeze all model parameters.
7. Prepare the official training and validation datasets.

The model is initially kept on the CPU.

The pilot extraction is performed on the CPU using
float32 precision.

GPU performance and memory requirements will be
evaluated separately before scaling up the experiments.

In [2]:
from pie.predict import load_checkpoint
from pie.data.datamodule import PieDataModule
from pie.model.pie import PieModel


# --------------------------------------------------
# Load pretrained checkpoint
# --------------------------------------------------

checkpoint_path = CHECKPOINT_DIR / "best_auprc.ckpt"

loaded = load_checkpoint(checkpoint_path)


# --------------------------------------------------
# Initialize data module
# --------------------------------------------------

dm = PieDataModule(
    cfg=loaded.config.data,
    run_dir=None,
    stats=loaded.stats,
    seed=loaded.config.seed,
    evidence_source=loaded.config.data,
)

dm.setup_stats(rank=0)
dm.setup(stage="fit")


# --------------------------------------------------
# Model inputs
# --------------------------------------------------

source_dims = dict(dm.source_dims)
gene_query_text = dm.gene_query_text()


# --------------------------------------------------
# Reconstruct pretrained model
# --------------------------------------------------

model = PieModel(
    cfg=loaded.config.model,
    stats=loaded.stats,
    source_dims=source_dims,
    gene_query_text=gene_query_text,
)

load_result = model.load_state_dict(
    loaded.state_dict,
    strict=True,
)


# --------------------------------------------------
# Freeze pretrained PIE
# --------------------------------------------------

model.requires_grad_(False)
model.eval()


# --------------------------------------------------
# Verify setup
# --------------------------------------------------

print("Pretrained PIE loaded successfully!")

print("\nCheckpoint verification:")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

print("\nArchitecture:")
print("Encoder:", len(model.encoder))
print("Processor:", len(model.processor))
print("Decoder:", len(model.decoder))

print("\nDatasets:")
print("Train conditions:", len(dm.train_dataset))
print("Validation conditions:", len(dm.val_dataset))

print("\nModel:")
print("Parameters frozen:", all(
    not p.requires_grad for p in model.parameters()
))
print("Device:", next(model.parameters()).device)

d:\bio_paper\pie-sae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pretrained PIE loaded successfully!

Checkpoint verification:
Missing keys: []
Unexpected keys: []

Architecture:
Encoder: 4
Processor: 6
Decoder: 4

Datasets:
Train conditions: 2603
Validation conditions: 290

Model:
Parameters frozen: True
Device: cpu


## 3. Activation Dataset Design

### Objective

We define the structure of the activation datasets used for
layer-wise Sparse Autoencoder training.

### Representation Types

PIE produces two distinct types of internal representations.

**Latent representations**

Encoder and processor blocks produce 512 latent vectors
per experimental condition.

Each latent vector has dimensionality 768.

**Gene-specific representations**

Decoder blocks produce representations associated with
individual output genes.

The selected checkpoint predicts responses for 6,642 genes.

### Dataset Structure

For N experimental conditions, the expected activation
shapes are:

- Encoder / Processor: (N, 512, 768)
- Decoder: (N, 6642, 768)

For SAE training, the first two dimensions can be flattened
into a collection of 768-dimensional vectors.

### Cross-layer Consistency

The same experimental conditions and corresponding positions
must be preserved across layers.

In particular:

- Latent slot indices are preserved across encoder and
  processor layers.
- Gene identities are preserved across decoder layers.
- Experimental condition identifiers are retained.

### Storage Strategy

We use HDF5 to store activations on disk.

This enables subsequent SAE training without loading the
complete activation dataset into RAM.

The initial experiment extracts and stores activations for
a single training condition.

In [3]:
import h5py
from collections import defaultdict

from pie.predict import _batch_to


# --------------------------------------------------
# Select one training condition
# --------------------------------------------------

train_loader = dm.train_dataloader()

sample = dm.train_dataset[0]
batch = train_loader.collate_fn([sample])

device = torch.device("cpu")

model.to(device)
model.eval()

batch_ready = _batch_to(
    batch,
    device,
    model.latents.dtype,
)


# --------------------------------------------------
# Activation collection
# --------------------------------------------------

activations = defaultdict(list)
handles = []


def make_hook(layer_name):
    """Capture the output of a Transformer block."""

    def hook(module, inputs, output):

        if not isinstance(output, torch.Tensor):
            raise TypeError(
                f"Expected Tensor at {layer_name}, "
                f"got {type(output)}"
            )

        activations[layer_name].append(
            output.detach().cpu().clone()
        )

    return hook


# Register hooks on all 14 blocks
for group_name in ["encoder", "processor", "decoder"]:

    blocks = getattr(model, group_name)

    for index, block in enumerate(blocks):

        name = f"{group_name}.{index}"

        handle = block.register_forward_hook(
            make_hook(name)
        )

        handles.append(handle)


# --------------------------------------------------
# Forward pass
# --------------------------------------------------

try:
    with torch.inference_mode():
        outputs = model(batch_ready)

finally:
    for handle in handles:
        handle.remove()


# --------------------------------------------------
# Verify extracted activations
# --------------------------------------------------

assert len(activations) == 14

print("Activation extraction completed.")
print("Observed layers:", len(activations))

for name, tensors in activations.items():

    # Decoder blocks are called once per gene chunk
    combined = torch.cat(tensors, dim=1)

    assert combined.shape[0] == 1
    assert combined.shape[-1] == 768
    assert torch.isfinite(combined).all()

    print(
        f"{name:<15} "
        f"calls={len(tensors)} "
        f"combined_shape={tuple(combined.shape)}"
    )

Activation extraction completed.
Observed layers: 14
encoder.0       calls=1 combined_shape=(1, 512, 768)
encoder.1       calls=1 combined_shape=(1, 512, 768)
encoder.2       calls=1 combined_shape=(1, 512, 768)
encoder.3       calls=1 combined_shape=(1, 512, 768)
processor.0     calls=1 combined_shape=(1, 512, 768)
processor.1     calls=1 combined_shape=(1, 512, 768)
processor.2     calls=1 combined_shape=(1, 512, 768)
processor.3     calls=1 combined_shape=(1, 512, 768)
processor.4     calls=1 combined_shape=(1, 512, 768)
processor.5     calls=1 combined_shape=(1, 512, 768)
decoder.0       calls=4 combined_shape=(1, 6642, 768)
decoder.1       calls=4 combined_shape=(1, 6642, 768)
decoder.2       calls=4 combined_shape=(1, 6642, 768)
decoder.3       calls=4 combined_shape=(1, 6642, 768)


In [4]:
# --------------------------------------------------
# Output directory
# --------------------------------------------------

ACTIVATION_DIR = DATA_ROOT / "activations"
ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)

output_path = ACTIVATION_DIR / "smoke_train_0000.h5"
# This is a disposable smoke-test file.
# It is intentionally recreated on each notebook run.
# The pilot train/validation datasets are not overwritten.

# --------------------------------------------------
# Save activations
# --------------------------------------------------

with h5py.File(output_path, "w") as f:

    # Metadata
    f.attrs["split"] = "train"
    f.attrs["condition_index"] = 0
    f.attrs["checkpoint_fold"] = "k562"
    f.attrs["hidden_dim"] = 768

    # Preserve gene identifiers
    assert len(batch.groups) == 1

    gene_ids = (
        batch.groups[0]
        .gene_ids
        .detach()
        .cpu()
        .numpy()
    )

    f.create_dataset(
        "gene_ids",
        data=gene_ids,
    )

    # Save activations by component and layer
    for group_name in ["encoder", "processor", "decoder"]:

        group = f.create_group(group_name)

        blocks = getattr(model, group_name)

        for index in range(len(blocks)):

            name = f"{group_name}.{index}"

            combined = torch.cat(
                activations[name],
                dim=1,
            )

            array = (
                combined
                .squeeze(0)
                .numpy()
                .astype(np.float32, copy=False)
            )

            group.create_dataset(
                str(index),
                data=array,
                compression="lzf",
                chunks=(
                    min(256, array.shape[0]),
                    array.shape[1],
                ),
            )


# --------------------------------------------------
# Verify saved file
# --------------------------------------------------

print("Activations saved successfully!")
print("File:", output_path)
print(
    "File size:",
    f"{output_path.stat().st_size / 1024**2:.2f} MiB",
)

with h5py.File(output_path, "r") as f:

    print("\nStored activation shapes:")

    for group_name in ["encoder", "processor", "decoder"]:

        for index in range(len(getattr(model, group_name))):

            shape = f[group_name][str(index)].shape

            print(
                f"{group_name}.{index}: {shape}"
            )

    print("\nStored gene IDs:", f["gene_ids"].shape)

Activations saved successfully!
File: D:\bio_paper\pie-sae\data\activations\smoke_train_0000.h5
File size: 93.02 MiB

Stored activation shapes:
encoder.0: (512, 768)
encoder.1: (512, 768)
encoder.2: (512, 768)
encoder.3: (512, 768)
processor.0: (512, 768)
processor.1: (512, 768)
processor.2: (512, 768)
processor.3: (512, 768)
processor.4: (512, 768)
processor.5: (512, 768)
decoder.0: (6642, 768)
decoder.1: (6642, 768)
decoder.2: (6642, 768)
decoder.3: (6642, 768)

Stored gene IDs: (6642,)


## 4. Activation Storage Verification

### Objective

Verify that all extracted PIE activations have been
stored correctly in HDF5 format.

### Verification Procedure

We compare the saved activation arrays against the
original tensors captured by PyTorch forward hooks.

The following properties are checked:

1. All 14 Transformer layers are present.
2. Activation shapes match the expected dimensions.
3. Stored values exactly match the original activations.
4. Gene identifiers are preserved.
5. Experimental metadata is consistent.

This verification ensures that the stored datasets can
be used for subsequent SAE training without introducing
numerical changes during serialization.

In [5]:
import h5py
import numpy as np
import torch

print("ACTIVATION STORAGE VERIFICATION")
print("=" * 60)

verified_layers = 0

with h5py.File(output_path, "r") as f:

    # Verify metadata
    assert f.attrs["split"] == "train"
    assert f.attrs["condition_index"] == 0
    assert f.attrs["hidden_dim"] == 768

    # Verify gene identifiers
    original_gene_ids = (
        batch.groups[0]
        .gene_ids
        .detach()
        .cpu()
        .numpy()
    )

    stored_gene_ids = f["gene_ids"][:]

    assert np.array_equal(
        original_gene_ids,
        stored_gene_ids,
    )

    print("Gene identifiers: verified")

    # Verify all Transformer activations
    for component in ["encoder", "processor", "decoder"]:

        blocks = getattr(model, component)

        for index in range(len(blocks)):

            layer_name = f"{component}.{index}"

            original = (
                torch.cat(
                    activations[layer_name],
                    dim=1,
                )
                .squeeze(0)
                .numpy()
            )

            stored = f[component][str(index)][:]

            assert original.shape == stored.shape

            assert np.array_equal(
                original,
                stored,
            ), f"Data mismatch: {layer_name}"

            assert np.isfinite(stored).all()

            verified_layers += 1

            print(f"{layer_name:<15} OK")

print("\n" + "=" * 60)
print(f"Verified layers: {verified_layers}/14")
print("All activation arrays match exactly")

ACTIVATION STORAGE VERIFICATION
Gene identifiers: verified
encoder.0       OK
encoder.1       OK
encoder.2       OK
encoder.3       OK
processor.0     OK
processor.1     OK
processor.2     OK
processor.3     OK
processor.4     OK
processor.5     OK
decoder.0       OK
decoder.1       OK
decoder.2       OK
decoder.3       OK

Verified layers: 14/14
All activation arrays match exactly


## 5. Pilot Layer-wise Activation Extraction

### Objective

We extend the single-condition activation extraction experiment
to multiple experimental conditions.

The objective is to construct reproducible activation datasets
for training and validating layer-wise Sparse Autoencoders.

### Experimental Design

The pilot experiment uses:

- 16 randomly selected training conditions.
- 4 randomly selected validation conditions.
- All 14 Transformer blocks.
- The pretrained PIE model in evaluation mode.
- Float32 activation storage.

Training and validation conditions are selected independently
from the official PIE dataset splits.

### Extraction Strategy

For each experimental condition:

1. Construct a batch using the official PIE collate function.
2. Execute the pretrained model without gradient tracking.
3. Capture the outputs of all Transformer blocks.
4. Concatenate decoder activations across gene chunks.
5. Save activations directly into HDF5 datasets.
6. Preserve experimental condition indices and gene identifiers.

Only one experimental condition is processed at a time.

### Reproducibility

The random seed, selected condition indices, checkpoint fold,
and activation tensor shapes are recorded.

The same conditions and positions are preserved across layers,
allowing subsequent cross-layer feature analysis.

### Scope

This is a pilot extraction experiment.

The resulting datasets will validate the extraction pipeline
and support preliminary SAE training.

Larger datasets will be generated after evaluating pilot results.

In [6]:
import math
import h5py
import numpy as np
import torch

from collections import defaultdict
from pathlib import Path

from pie.predict import _batch_to


# --------------------------------------------------
# Experiment configuration
# --------------------------------------------------

N_TRAIN = 16
N_VAL = 4

DEVICE = torch.device("cpu")

HIDDEN_DIM = model.cfg.d_model
N_LATENTS = model.cfg.n_latents
N_GENES = gene_query_text.shape[0]

EXTRACTION_DIR = DATA_ROOT / "activations" / "pilot"
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# Prepare model
# --------------------------------------------------

model = model.to(DEVICE)
model.requires_grad_(False)
model.eval()


# --------------------------------------------------
# Layer names and expected dimensions
# --------------------------------------------------

layer_names = []

for component in ["encoder", "processor", "decoder"]:

    blocks = getattr(model, component)

    for index in range(len(blocks)):
        layer_names.append(f"{component}.{index}")


assert len(layer_names) == 14


def expected_positions(layer_name):

    if layer_name.startswith("decoder"):
        return N_GENES

    return N_LATENTS


# --------------------------------------------------
# Activation collection
# --------------------------------------------------

captured = defaultdict(list)
hook_handles = []


def make_hook(layer_name):

    def hook(module, inputs, output):

        if not isinstance(output, torch.Tensor):
            raise TypeError(
                f"Unexpected output at {layer_name}: "
                f"{type(output)}"
            )

        captured[layer_name].append(
            output.detach().cpu().clone()
        )

    return hook





# --------------------------------------------------
# Streaming extraction
# --------------------------------------------------

def extract_split(
    dataset,
    collate_fn,
    condition_indices,
    split_name,
    output_path,
):

    condition_indices = np.asarray(
        condition_indices,
        dtype=np.int64,
    )

    n_conditions = len(condition_indices)

    # Do not silently overwrite previous results
    if output_path.exists():
        raise FileExistsError(
            f"Output already exists: {output_path}"
        )

    # Keep incomplete results separate
    partial_path = output_path.with_suffix(".partial.h5")

    if partial_path.exists():
        raise FileExistsError(
            f"Incomplete file already exists: {partial_path}"
        )

    print(f"\nExtracting {split_name}")
    print(f"Conditions: {n_conditions}")
    print(f"Output: {output_path}")

    with h5py.File(partial_path, "w") as f:

        # ------------------------------------------
        # Metadata
        # ------------------------------------------

        f.attrs["split"] = split_name
        f.attrs["checkpoint_fold"] = "k562"
        f.attrs["seed"] = SEED
        f.attrs["hidden_dim"] = HIDDEN_DIM
        f.attrs["n_conditions"] = n_conditions
        f.attrs["completed_count"] = 0

        f.create_dataset(
            "condition_indices",
            data=condition_indices,
        )

        f.create_dataset(
            "gene_ids",
            shape=(n_conditions, N_GENES),
            dtype=np.int64,
            chunks=(1, N_GENES),
            compression="lzf",
        )

        # ------------------------------------------
        # Create activation datasets
        # ------------------------------------------

        activation_datasets = {}

        for layer_name in layer_names:

            component, index = layer_name.split(".")

            if component not in f:
                f.create_group(component)

            n_positions = expected_positions(layer_name)

            dataset_handle = f[component].create_dataset(
                index,
                shape=(
                    n_conditions,
                    n_positions,
                    HIDDEN_DIM,
                ),
                dtype=np.float32,
                chunks=(
                    1,
                    min(256, n_positions),
                    HIDDEN_DIM,
                ),
                compression="lzf",
            )

            activation_datasets[layer_name] = dataset_handle

        # ------------------------------------------
        # Process conditions sequentially
        # ------------------------------------------

        for position, condition_index in enumerate(
            condition_indices
        ):

            sample = dataset[int(condition_index)]

            batch = collate_fn([sample])

            if len(batch.groups) != 1:
                raise ValueError(
                    "Expected one gene group for "
                    "a single-condition batch."
                )

            batch_ready = _batch_to(
                batch,
                DEVICE,
                model.latents.dtype,
            )

            captured.clear()

            with torch.inference_mode():
                _ = model(batch_ready)

            # Verify all expected layers were captured
            assert set(captured.keys()) == set(layer_names)

            # Save gene identifiers for this condition
            gene_ids = (
                batch.groups[0]
                .gene_ids
                .detach()
                .cpu()
                .numpy()
            )

            assert gene_ids.shape == (N_GENES,)

            f["gene_ids"][position] = gene_ids

            # Save all layer activations
            for layer_name in layer_names:

                tensors = captured[layer_name]

                expected_calls = (
                    math.ceil(
                        N_GENES / model.cfg.inference_chunk_size
                    )
                    if layer_name.startswith("decoder")
                    else 1
                )

                assert len(tensors) == expected_calls, (
                    f"Unexpected hook calls: {layer_name}"
                )

                combined = torch.cat(tensors, dim=1)

                assert combined.shape == (
                    1,
                    expected_positions(layer_name),
                    HIDDEN_DIM,
                )

                if not torch.isfinite(combined).all():
                    raise ValueError(
                        f"Non-finite activations: {layer_name}"
                    )

                array = combined.squeeze(0).numpy()

                activation_datasets[layer_name][position] = array

            # Release captured tensors
            captured.clear()

            # Update progress metadata
            f.attrs["completed_count"] = position + 1
            f.flush()

            print(
                f"{split_name}: "
                f"{position + 1}/{n_conditions}",
                end="\r",
                flush=True,
            )

    # Mark the file as complete
    partial_path.replace(output_path)

    print(f"\n{split_name} extraction completed!")
    print(
        "File size:",
        f"{output_path.stat().st_size / 1024**3:.2f} GiB"
    )

In [7]:
# --------------------------------------------------
# Reproducible condition sampling
# --------------------------------------------------

rng_train = np.random.default_rng(SEED)
rng_val = np.random.default_rng(SEED + 1)

train_indices = np.sort(
    rng_train.choice(
        len(dm.train_dataset),
        size=N_TRAIN,
        replace=False,
    )
)

val_indices = np.sort(
    rng_val.choice(
        len(dm.val_dataset),
        size=N_VAL,
        replace=False,
    )
)

print("Selected training indices:", train_indices)
print("Selected validation indices:", val_indices)


# --------------------------------------------------
# Official collate function
# --------------------------------------------------

collate_fn = dm.train_dataloader().collate_fn


# --------------------------------------------------
# Output paths
# --------------------------------------------------

train_path = EXTRACTION_DIR / "train_pilot_16.h5"
val_path = EXTRACTION_DIR / "val_pilot_4.h5"


# --------------------------------------------------
# Verify existing extraction files
# --------------------------------------------------

def verify_existing_file(path, split_name, indices):
    """Verify that an existing extraction matches this run."""

    indices = np.asarray(indices, dtype=np.int64)
    n_conditions = len(indices)

    with h5py.File(path, "r") as f:

        assert f.attrs["split"] == split_name
        assert f.attrs["n_conditions"] == n_conditions
        assert f.attrs["completed_count"] == n_conditions
        assert f.attrs["hidden_dim"] == HIDDEN_DIM

        assert np.array_equal(
            f["condition_indices"][:],
            indices,
        )

        assert f["gene_ids"].shape == (
            n_conditions,
            N_GENES,
        )

        for layer_name in layer_names:

            component, index = layer_name.split(".")

            dataset = f[component][index]

            expected_shape = (
                n_conditions,
                expected_positions(layer_name),
                HIDDEN_DIM,
            )

            assert dataset.shape == expected_shape
            assert dataset.dtype == np.float32

            # Check representative stored values
            sample = dataset[0, :32, :]

            assert np.isfinite(sample).all()

    print(f"Verified existing file: {path.name}")


# --------------------------------------------------
# Prepare extraction jobs
# --------------------------------------------------

jobs = [
    (
        "train",
        dm.train_dataset,
        train_indices,
        train_path,
    ),
    (
        "validation",
        dm.val_dataset,
        val_indices,
        val_path,
    ),
]

pending_jobs = []


# Remove hooks left from previous executions
for handle in hook_handles:
    handle.remove()

hook_handles.clear()
captured.clear()


for split_name, dataset, indices, path in jobs:

    if path.exists():

        verify_existing_file(
            path,
            split_name,
            indices,
        )

    else:

        pending_jobs.append(
            (split_name, dataset, indices, path)
        )


# --------------------------------------------------
# Extract only missing datasets
# --------------------------------------------------

if pending_jobs:

    # Register fresh forward hooks
    for component in ["encoder", "processor", "decoder"]:

        for index, block in enumerate(
            getattr(model, component)
        ):

            name = f"{component}.{index}"

            handle = block.register_forward_hook(
                make_hook(name)
            )

            hook_handles.append(handle)

    try:

        for split_name, dataset, indices, path in pending_jobs:

            extract_split(
                dataset=dataset,
                collate_fn=collate_fn,
                condition_indices=indices,
                split_name=split_name,
                output_path=path,
            )

    finally:

        # Always remove forward hooks
        for handle in hook_handles:
            handle.remove()

        hook_handles.clear()
        captured.clear()

else:

    print("\nAll extraction files already exist.")
    print("No repeated inference required.")


print("\nPilot extraction verification completed.")

Selected training indices: [ 222  230  244  523 1122 1137 1367 1695 1809 1866 1912 1979 2003 2046
 2226 2535]
Selected validation indices: [ 12 115 145 187]
Verified existing file: train_pilot_16.h5
Verified existing file: val_pilot_4.h5

All extraction files already exist.
No repeated inference required.

Pilot extraction verification completed.


In [8]:
import h5py
import numpy as np

# Existing activation datasets
files = {
    "train": (train_path, 16),
    "validation": (val_path, 4),
}

for split_name, (path, expected_n) in files.items():

    print(f"\nValidating {split_name}: {path.name}")

    assert path.is_file(), f"File not found: {path}"

    with h5py.File(path, "r") as f:

        # Check metadata
        assert f.attrs["n_conditions"] == expected_n
        assert f.attrs["completed_count"] == expected_n
        assert f.attrs["split"] == split_name

        assert f["condition_indices"].shape == (expected_n,)
        assert f["gene_ids"].shape == (expected_n, 6642)

        # Check all Transformer blocks
        verified = 0

        for component, n_layers, n_positions in [
            ("encoder", 4, 512),
            ("processor", 6, 512),
            ("decoder", 4, 6642),
        ]:

            for index in range(n_layers):

                dataset = f[component][str(index)]

                expected_shape = (
                    expected_n,
                    n_positions,
                    768,
                )

                assert dataset.shape == expected_shape
                assert dataset.dtype == np.float32

                # Check representative stored values
                sample = dataset[0, :32, :]

                assert np.isfinite(sample).all()

                verified += 1

        print("Conditions:", expected_n)
        print("Verified layers:", verified)
        print("Completed:", True)

print("\nBoth pilot datasets validated successfully!")


Validating train: train_pilot_16.h5
Conditions: 16
Verified layers: 14
Completed: True

Validating validation: val_pilot_4.h5
Conditions: 4
Verified layers: 14
Completed: True

Both pilot datasets validated successfully!


## 6. Expanded Activation Dataset

We extend the validated activation extraction pipeline to
a larger sample of experimental conditions.

### Configuration

- Training conditions: 128
- Validation conditions: 32
- Transformer blocks: 14
- Hidden dimension: 768
- Sampling seed: 42

The expanded datasets are stored separately from the
original pilot files.

The same pretrained PIE checkpoint, official dataset
splits, forward hooks, and HDF5 extraction procedure
are used.

This dataset supports the next stage of layer-wise
SAE training and evaluation.

In [9]:
# --------------------------------------------------
# Expanded extraction configuration
# --------------------------------------------------

N_TRAIN_EXPANDED = 128
N_VAL_EXPANDED = 32

EXPANDED_DIR = (
    DATA_ROOT
    / "activations"
    / "expanded_128_32"
)

EXPANDED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Reproducible sampling
# --------------------------------------------------

rng_train = np.random.default_rng(SEED)
rng_val = np.random.default_rng(SEED + 1)

expanded_train_indices = np.sort(
    rng_train.choice(
        len(dm.train_dataset),
        size=N_TRAIN_EXPANDED,
        replace=False,
    )
)

expanded_val_indices = np.sort(
    rng_val.choice(
        len(dm.val_dataset),
        size=N_VAL_EXPANDED,
        replace=False,
    )
)


# --------------------------------------------------
# Output paths
# --------------------------------------------------

expanded_train_path = (
    EXPANDED_DIR / "train_128.h5"
)

expanded_val_path = (
    EXPANDED_DIR / "val_32.h5"
)


# --------------------------------------------------
# Prepare jobs
# --------------------------------------------------

expanded_jobs = [
    (
        "train",
        dm.train_dataset,
        expanded_train_indices,
        expanded_train_path,
    ),
    (
        "validation",
        dm.val_dataset,
        expanded_val_indices,
        expanded_val_path,
    ),
]

pending_jobs = []


# Remove any previously registered extraction hooks
for handle in hook_handles:
    handle.remove()

hook_handles.clear()
captured.clear()


# --------------------------------------------------
# Verify existing files or schedule extraction
# --------------------------------------------------

for split_name, dataset, indices, path in expanded_jobs:

    if path.exists():

        verify_existing_file(
            path,
            split_name,
            indices,
        )

    else:

        partial_path = path.with_suffix(".partial.h5")

        if partial_path.exists():
            raise FileExistsError(
                f"Incomplete extraction exists: {partial_path}"
            )

        pending_jobs.append(
            (split_name, dataset, indices, path)
        )


# --------------------------------------------------
# Extract missing datasets
# --------------------------------------------------

if pending_jobs:

    try:

        # Register hooks
        for component in ["encoder", "processor", "decoder"]:

            for index, block in enumerate(
                getattr(model, component)
            ):

                layer_name = f"{component}.{index}"

                handle = block.register_forward_hook(
                    make_hook(layer_name)
                )

                hook_handles.append(handle)

        # Use the existing extraction function
        collate_fn = dm.train_dataloader().collate_fn

        for split_name, dataset, indices, path in pending_jobs:

            extract_split(
                dataset=dataset,
                collate_fn=collate_fn,
                condition_indices=indices,
                split_name=split_name,
                output_path=path,
            )

    finally:

        for handle in hook_handles:
            handle.remove()

        hook_handles.clear()
        captured.clear()

else:

    print("All expanded datasets already exist.")


# --------------------------------------------------
# Final verification
# --------------------------------------------------

print("\nVerifying expanded extraction...")

for split_name, dataset, indices, path in expanded_jobs:

    verify_existing_file(
        path,
        split_name,
        indices,
    )

    print(
        f"{split_name}: "
        f"{len(indices)} conditions, "
        f"{path.stat().st_size / 1024**3:.2f} GiB"
    )

print("\nExpanded extraction completed successfully!")


Extracting train
Conditions: 128
Output: D:\bio_paper\pie-sae\data\activations\expanded_128_32\train_128.h5
train: 128/128
train extraction completed!
File size: 11.62 GiB

Extracting validation
Conditions: 32
Output: D:\bio_paper\pie-sae\data\activations\expanded_128_32\val_32.h5
validation: 32/32
validation extraction completed!
File size: 2.90 GiB

Verifying expanded extraction...
Verified existing file: train_128.h5
train: 128 conditions, 11.62 GiB
Verified existing file: val_32.h5
validation: 32 conditions, 2.90 GiB

Expanded extraction completed successfully!


## 7. Gene Identity and Metadata Verification

In [9]:
import json
import h5py
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "external" / "pie").is_dir()
HF_ROOT = PROJECT_ROOT / "data" / "hf"

# --------------------------------------------------
# 1. Find the PIE gene vocabulary
# --------------------------------------------------

candidates = []

for path in HF_ROOT.rglob("meta.json"):
    with open(path, encoding="utf-8") as f:
        metadata = json.load(f)

    if (
        isinstance(metadata.get("genes"), list)
        and len(metadata["genes"]) == 6642
    ):
        candidates.append((path, metadata))

assert len(candidates) == 1, (
    f"Expected one gene vocabulary, found {len(candidates)}"
)

metadata_path, metadata = candidates[0]
genes = metadata["genes"]

assert all(isinstance(gene, str) for gene in genes)
assert len(set(genes)) == 6642

# --------------------------------------------------
# 2. Verify HDF5 gene indices
# --------------------------------------------------

data_dir = (
    PROJECT_ROOT
    / "data"
    / "activations"
    / "expanded_128_32"
)

expected_ids = np.arange(6642, dtype=np.int64)

for filename in ["train_128.h5", "val_32.h5"]:

    with h5py.File(data_dir / filename, "r") as f:
        stored_ids = f["gene_ids"][:]

    assert np.all(stored_ids == expected_ids[None, :])

    print(f"{filename}: gene indices verified")

# --------------------------------------------------
# 3. Save the gene vocabulary
# --------------------------------------------------

output_dir = PROJECT_ROOT / "results" / "gene_reference"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "gene_index_to_symbol.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "source_metadata": str(
                metadata_path.relative_to(PROJECT_ROOT)
            ),
            "n_genes": len(genes),
            "genes": genes,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------
# 4. Display examples
# --------------------------------------------------

print("\nFirst 10 gene mappings:")

for index, symbol in enumerate(genes[:10]):
    print(f"{index:4d} -> {symbol}")

print("\nTotal genes:", len(genes))
print("Mapping saved to:", output_path)
print("\nGene mapping verification completed.")

train_128.h5: gene indices verified
val_32.h5: gene indices verified

First 10 gene mappings:
   0 -> NOC2L
   1 -> HES4
   2 -> ISG15
   3 -> SDF4
   4 -> B3GALT6
   5 -> UBE2J2
   6 -> ACAP3
   7 -> PUSL1
   8 -> INTS11
   9 -> DVL1

Total genes: 6642
Mapping saved to: D:\bio_paper\pie-sae\results\gene_reference\gene_index_to_symbol.json

Gene mapping verification completed.


### 7.1. Verification Against the PIE Source Code

We verify how gene identifiers are constructed and propagated
through the official PIE data pipeline.

The goal is to establish that the decoder gene positions
correspond to the ordered gene vocabulary stored in the
preprocessed dataset metadata.

This verification is performed directly against the pinned
PIE source code rather than inferring gene identity solely
from matching array dimensions.

In [ ]:
from pathlib import Path
import re

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "external" / "pie").is_dir()

PIE_SRC = (
    PROJECT_ROOT
    / "external"
    / "pie"
    / "src"
    / "pie"
)

assert PIE_SRC.is_dir(), PIE_SRC

# --------------------------------------------------
# Relevant gene-axis operations
# --------------------------------------------------

patterns = [
    r"\bgene_ids\b",
    r"\bgene_to_id\b",
    r"\bgene_query_text\b",
    r"\bgenes\b",
]

regex = re.compile("|".join(patterns))

# --------------------------------------------------
# Search the official PIE implementation
# --------------------------------------------------

matches = []

for path in sorted(PIE_SRC.rglob("*.py")):

    # Focus on dataset construction and model inputs
    if (
        "data" not in path.parts
        and path.name not in {"pie.py", "predict.py"}
    ):
        continue

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    for line_number, line in enumerate(lines):

        if regex.search(line):

            matches.append((
                path.relative_to(PIE_SRC),
                line_number,
                lines,
            ))

print("=" * 65)
print("PIE GENE-AXIS SOURCE INSPECTION")
print("=" * 65)

print("\nTotal matching locations:", len(matches))

# Print a limited number of locations with context
for path, line_number, lines in matches[:35]:

    print("\n" + "-" * 65)
    print(f"{path}:{line_number + 1}")

    start = max(0, line_number - 2)
    end = min(len(lines), line_number + 3)

    for i in range(start, end):
        print(f"{i + 1:4d} | {lines[i]}")

print("\nSource inspection completed.")

PIE GENE-AXIS SOURCE INSPECTION

Total matching locations: 62

-----------------------------------------------------------------
data\datamodule.py:220
 218 |         union: dict[str, int] = {}
 219 |         for d in self.dirs:
 220 |             for gene in d.genes:
 221 |                 union.setdefault(gene, len(union))
 222 |         self.gene_axis = list(union)

-----------------------------------------------------------------
data\datamodule.py:224
 222 |         self.gene_axis = list(union)
 223 |         self.gene_union_ids = [
 224 |             np.asarray([union[gene] for gene in d.genes], dtype=np.int64) for d in self.dirs
 225 |         ]
 226 |         source_paths = {

-----------------------------------------------------------------
data\datamodule.py:343
 341 |     # --- rows and datasets ----------------------------------------------------------------
 342 | 
 343 |     def gene_query_text(self) -> torch.Tensor:
 344 |         """(len(gene_axis), gene_query_dim) floa

In [ ]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "external" / "pie").is_dir()

# Set PIE paths before importing its modules.
os.environ["PIE_DATA_ROOT"] = str(PROJECT_ROOT / "data")
os.environ["PIE_CACHE_DIR"] = str(PROJECT_ROOT / "cache")
os.environ["PIE_RUNS_ROOT"] = str(PROJECT_ROOT / "runs")

PIE_SRC = PROJECT_ROOT / "external" / "pie" / "src"

if str(PIE_SRC) not in sys.path:
    sys.path.insert(0, str(PIE_SRC))

from pie.predict import load_checkpoint
from pie.data.datamodule import PieDataModule


# --------------------------------------------------
# 1. Restore the actual PIE data configuration
# --------------------------------------------------

checkpoint_path = (
    PROJECT_ROOT
    / "models"
    / "replogle_wdataset"
    / "k562"
    / "best_auprc.ckpt"
)

assert checkpoint_path.is_file()

loaded = load_checkpoint(checkpoint_path)

dm_check = PieDataModule(
    cfg=loaded.config.data,
    run_dir=None,
    stats=loaded.stats,
    seed=loaded.config.seed,
    evidence_source=loaded.config.data,
)

dm_check.setup_stats(rank=0)
dm_check.setup(stage="fit")


# --------------------------------------------------
# 2. Inspect the actual gene axis
# --------------------------------------------------

actual_gene_axis = list(dm_check.gene_axis)

print("PIE dataset directories:", len(dm_check.dirs))
print("Actual gene axis length:", len(actual_gene_axis))
print("First 10 genes:", actual_gene_axis[:10])

for index, directory in enumerate(dm_check.dirs):
    print(
        f"Dataset {index}:",
        len(directory.genes),
        "genes",
    )


# --------------------------------------------------
# 3. Compare with our saved gene vocabulary
# --------------------------------------------------

mapping_path = (
    PROJECT_ROOT
    / "results"
    / "gene_reference"
    / "gene_index_to_symbol.json"
)

with open(mapping_path, encoding="utf-8") as f:
    saved_mapping = json.load(f)

saved_genes = saved_mapping["genes"]

print("\nSaved vocabulary length:", len(saved_genes))

same_order = actual_gene_axis == saved_genes

print("Exact gene ordering match:", same_order)

if not same_order:
    mismatches = [
        (i, a, b)
        for i, (a, b) in enumerate(
            zip(actual_gene_axis, saved_genes)
        )
        if a != b
    ]

    print("First mismatches:", mismatches[:10])

assert len(actual_gene_axis) == 6642
assert same_order, "Gene vocabulary ordering mismatch!"

print("\nPIE gene-axis verification PASSED.")

d:\bio_paper\pie-sae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PIE dataset directories: 1
Actual gene axis length: 6642
First 10 genes: ['NOC2L', 'HES4', 'ISG15', 'SDF4', 'B3GALT6', 'UBE2J2', 'ACAP3', 'PUSL1', 'INTS11', 'DVL1']
Dataset 0: 6642 genes

Saved vocabulary length: 6642
Exact gene ordering match: True

PIE gene-axis verification PASSED.


## 7.2. Full HDF5 Activation Validation

We validate the complete expanded activation datasets
without repeating PIE inference.

The validation checks:

- HDF5 structure and metadata.
- Expected activation shapes and float32 data types.
- All activation values for NaN and Inf.
- Condition indices and decoder gene positions.
- Completion status of each extraction.

Each dataset is read sequentially, one experimental
condition at a time.

A machine-readable validation report is saved for
reproducibility.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "external" / "pie").is_dir()

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "activations"
    / "expanded_128_32"
)

FILES = {
    "train": (DATA_DIR / "train_128.h5", 128),
    "validation": (DATA_DIR / "val_32.h5", 32),
}

LAYER_COUNTS = {
    "encoder": 4,
    "processor": 6,
    "decoder": 4,
}

EXPECTED_POSITIONS = {
    "encoder": 512,
    "processor": 512,
    "decoder": 6642,
}

HIDDEN_DIM = 768
N_GENES = 6642

report = {
    "validation_type": "full_hdf5_activation_scan",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "datasets": {},
}


# --------------------------------------------------
# Validate every stored activation
# --------------------------------------------------

start_time = time.perf_counter()

for split_name, (path, expected_conditions) in FILES.items():

    print(f"\nValidating {split_name}: {path.name}")

    assert path.is_file(), path

    split_result = {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "conditions": expected_conditions,
        "layers": {},
    }

    with h5py.File(path, "r") as f:

        assert f.attrs["split"] == split_name
        assert f.attrs["n_conditions"] == expected_conditions
        assert f.attrs["completed_count"] == expected_conditions

        condition_indices = f["condition_indices"][:]
        gene_ids = f["gene_ids"][:]

        assert condition_indices.shape == (expected_conditions,)
        assert len(np.unique(condition_indices)) == expected_conditions

        assert gene_ids.shape == (expected_conditions, N_GENES)

        assert np.all(
            gene_ids == np.arange(N_GENES)[None, :]
        )

        for component, n_layers in LAYER_COUNTS.items():

            for index in range(n_layers):

                name = f"{component}.{index}"

                dataset = f[component][str(index)]

                expected_shape = (
                    expected_conditions,
                    EXPECTED_POSITIONS[component],
                    HIDDEN_DIM,
                )

                assert dataset.shape == expected_shape, name
                assert dataset.dtype == np.float32, name

                checked_values = 0

                # Read every condition, not just a sample.
                for condition in range(expected_conditions):

                    activations = dataset[condition, :, :]

                    if not np.isfinite(activations).all():
                        raise ValueError(
                            f"Non-finite activations: "
                            f"{split_name}, {name}, "
                            f"condition {condition}"
                        )

                    checked_values += activations.size

                split_result["layers"][name] = {
                    "shape": list(dataset.shape),
                    "dtype": str(dataset.dtype),
                    "checked_values": checked_values,
                    "all_finite": True,
                }

            print(
                f"  {component}: "
                f"{n_layers}/{n_layers} layers validated"
            )

    report["datasets"][split_name] = split_result

    print(f"{split_name}: PASSED")


# --------------------------------------------------
# Save validation report
# --------------------------------------------------

REPORT_DIR = (
    PROJECT_ROOT
    / "results"
    / "extraction_validation"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

report["elapsed_seconds"] = (
    time.perf_counter() - start_time
)

report["status"] = "passed"

report_path = REPORT_DIR / "full_hdf5_validation.json"

with open(report_path, "w", encoding="utf-8") as f:

    json.dump(
        report,
        f,
        indent=2,
    )


# --------------------------------------------------
# Final summary
# --------------------------------------------------

assert len(report["datasets"]) == 2

for split_result in report["datasets"].values():
    assert len(split_result["layers"]) == 14

print("\n" + "=" * 60)
print("FULL HDF5 VALIDATION PASSED")
print("=" * 60)

print("Datasets validated:", len(report["datasets"]))
print("Layers per dataset:", 14)
print("All activation values finite: True")

print(
    "Elapsed seconds:",
    round(report["elapsed_seconds"], 2)
)

print("Report saved to:", report_path)


Validating train: train_128.h5
  encoder: 4/4 layers validated
  processor: 6/6 layers validated
  decoder: 4/4 layers validated
train: PASSED

Validating validation: val_32.h5
  encoder: 4/4 layers validated
  processor: 6/6 layers validated
  decoder: 4/4 layers validated
validation: PASSED

FULL HDF5 VALIDATION PASSED
Datasets validated: 2
Layers per dataset: 14
All activation values finite: True
Elapsed seconds: 43.71
Report saved to: D:\bio_paper\pie-sae\results\extraction_validation\full_hdf5_validation.json


## 7.3. Extraction Reproducibility Manifest

We record the provenance and configuration of the expanded
PIE activation datasets.

The manifest contains:

- PIE source code revision.
- Pretrained model checkpoint identity.
- Training and validation condition indices.
- Random sampling seeds.
- HDF5 dataset metadata.
- Gene vocabulary reference.
- Full activation validation report.

The manifest allows the extraction configuration to be
reconstructed without embedding large model or activation
files in the Git repository.
The legacy HDF5 seed attribute records the global extraction seed (42). The actual condition-sampling seeds are 42 for training and 43 for validation, as recorded and verified in the extraction manifest.

In [ ]:
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import h5py
import numpy as np


# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "external" / "pie").is_dir()

PIE_ROOT = PROJECT_ROOT / "external" / "pie"

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "models"
    / "replogle_wdataset"
    / "k562"
    / "best_auprc.ckpt"
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "activations"
    / "expanded_128_32"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "results"
    / "extraction_validation"
)

GENE_REFERENCE = (
    PROJECT_ROOT
    / "results"
    / "gene_reference"
    / "gene_index_to_symbol.json"
)

VALIDATION_REPORT = (
    REPORT_DIR / "full_hdf5_validation.json"
)

assert CHECKPOINT_PATH.is_file()
assert GENE_REFERENCE.is_file()
assert VALIDATION_REPORT.is_file()

assert "dm_check" in globals(), (
    "Run the preceding PIE gene-axis verification cell first."
)


# --------------------------------------------------
# 2. Source code version
# --------------------------------------------------

pie_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(PIE_ROOT),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

print("PIE commit:", pie_commit)


# --------------------------------------------------
# 3. File hashing
# --------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
gene_reference_sha256 = sha256_file(GENE_REFERENCE)

print("Checkpoint SHA256:", checkpoint_sha256)


# --------------------------------------------------
# 4. Verify deterministic condition sampling
# --------------------------------------------------

sampling = {
    "train": {
        "file": "train_128.h5",
        "seed": 42,
        "dataset": dm_check.train_dataset,
        "expected_conditions": 128,
    },
    "validation": {
        "file": "val_32.h5",
        "seed": 43,
        "dataset": dm_check.val_dataset,
        "expected_conditions": 32,
    },
}

dataset_records = {}

for split, config in sampling.items():

    path = DATA_DIR / config["file"]

    assert path.is_file()

    dataset_size = len(config["dataset"])
    n = config["expected_conditions"]

    # Recreate the original sampling procedure.
    rng = np.random.default_rng(config["seed"])

    expected_indices = np.sort(
        rng.choice(
            dataset_size,
            size=n,
            replace=False,
        )
    )

    with h5py.File(path, "r") as f:

        stored_indices = f["condition_indices"][:]

        assert np.array_equal(
            stored_indices,
            expected_indices,
        ), f"Sampling mismatch: {split}"

        assert f.attrs["completed_count"] == n

        attributes = {}

        for key in f.attrs.keys():

            value = f.attrs[key]

            if isinstance(value, np.generic):
                value = value.item()

            if isinstance(value, bytes):
                value = value.decode("utf-8")

            attributes[key] = value

    dataset_records[split] = {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "dataset_size": dataset_size,
        "sampling_seed": config["seed"],
        "condition_indices": stored_indices.tolist(),
        "n_conditions": n,
        "file_size_bytes": path.stat().st_size,
        "hdf5_attributes": attributes,
    }

    print(
        f"{split}: "
        f"{n} condition indices reproduced exactly"
    )


# --------------------------------------------------
# 5. Gene vocabulary
# --------------------------------------------------

with open(GENE_REFERENCE, encoding="utf-8") as f:
    gene_reference = json.load(f)

assert gene_reference["n_genes"] == 6642
assert gene_reference["genes"] == list(dm_check.gene_axis)


# --------------------------------------------------
# 6. Verify previous validation report
# --------------------------------------------------

with open(VALIDATION_REPORT, encoding="utf-8") as f:
    validation_report = json.load(f)

assert validation_report["status"] == "passed"


# --------------------------------------------------
# 7. Create manifest
# --------------------------------------------------

manifest = {
    "experiment": "PIE layer-wise activation extraction",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model": {
        "architecture": "PIE",
        "fold": "k562",
        "checkpoint": str(
            CHECKPOINT_PATH.relative_to(PROJECT_ROOT)
        ),
        "checkpoint_sha256": checkpoint_sha256,
        "pie_git_commit": pie_commit,
    },

    "extraction": {
        "components": {
            "encoder": 4,
            "processor": 6,
            "decoder": 4,
        },
        "hidden_dim": 768,
        "encoder_positions": 512,
        "processor_positions": 512,
        "decoder_positions": 6642,
        "activation_dtype": "float32",
        "storage_format": "HDF5",
        "datasets": dataset_records,
    },

    "gene_reference": {
        "file": str(
            GENE_REFERENCE.relative_to(PROJECT_ROOT)
        ),
        "sha256": gene_reference_sha256,
        "n_genes": 6642,
    },

    "validation": {
        "report": str(
            VALIDATION_REPORT.relative_to(PROJECT_ROOT)
        ),
        "status": validation_report["status"],
    },
}


# --------------------------------------------------
# 8. Save
# --------------------------------------------------

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

manifest_path = (
    REPORT_DIR / "extraction_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------
# 9. Final verification
# --------------------------------------------------

with open(manifest_path, encoding="utf-8") as f:
    saved_manifest = json.load(f)

assert saved_manifest["model"]["pie_git_commit"] == pie_commit

assert len(
    saved_manifest["extraction"]["datasets"]["train"][
        "condition_indices"
    ]
) == 128

assert len(
    saved_manifest["extraction"]["datasets"]["validation"][
        "condition_indices"
    ]
) == 32

print("\n" + "=" * 60)
print("EXTRACTION MANIFEST VERIFIED")
print("=" * 60)

print("PIE commit:", pie_commit)
print("Checkpoint checksum:", checkpoint_sha256[:16], "...")
print("Train conditions: 128")
print("Validation conditions: 32")
print("Gene vocabulary: 6642 genes")
print("Full HDF5 validation: PASSED")

print("\nManifest saved to:", manifest_path)

PIE commit: 21a526add9ff3a81b4f962d9b16bbae570052c63
Checkpoint SHA256: b9d5a0b3943666d892a56e7a9368f7850826b3f1fe6879e63a1a590f8add7d84
train: 128 condition indices reproduced exactly
validation: 32 condition indices reproduced exactly

EXTRACTION MANIFEST VERIFIED
PIE commit: 21a526add9ff3a81b4f962d9b16bbae570052c63
Checkpoint checksum: b9d5a0b3943666d8 ...
Train conditions: 128
Validation conditions: 32
Gene vocabulary: 6642 genes
Full HDF5 validation: PASSED

Manifest saved to: D:\bio_paper\pie-sae\results\extraction_validation\extraction_manifest.json


## 8. Conclusions

### 8.1. Activation Extraction

We implemented and validated an activation extraction pipeline for the pretrained PIE model, adapting the layer-wise extraction methodology used in Kendiukhov's single-cell SAE research.

The extraction covers all 14 principal Transformer blocks:

- 4 encoder cross-attention blocks;
- 6 processor self-attention blocks;
- 4 decoder cross-attention blocks.

All extracted representations have a hidden dimension of 768.

Encoder and processor representations contain 512 latent positions per experimental condition, whereas decoder representations contain 6642 gene-specific positions.

Forward hooks were used to capture block outputs without modifying the model's predictions.

### 8.2. Extracted Datasets

Two extraction stages were completed:

**Initial pilot:**
- 16 training conditions;
- 4 validation conditions.

**Expanded dataset:**
- 128 training conditions;
- 32 validation conditions.

The expanded dataset contains approximately 14.52 GiB of float32 activations stored in HDF5 format.

The experiments use the pretrained PIE checkpoint associated with the held-out K562 context configuration.

### 8.3. Gene Identity Verification

The decoder gene axis was verified against the official PIE data pipeline.

The `PieDataModule` constructs a gene vocabulary containing 6642 genes, and its ordering exactly matches the gene vocabulary stored in the dataset metadata.

The extracted HDF5 datasets contain gene indices from 0 to 6641 in a consistent order.

The gene-index-to-symbol mapping was saved to:

`results/gene_reference/gene_index_to_symbol.json`

This provides the gene identity reference required for subsequent decoder SAE feature analysis.

### 8.4. Data Integrity and Reproducibility

A full sequential validation of the expanded HDF5 datasets confirmed:

- Expected activation shapes for all 14 blocks.
- Float32 activation storage.
- Complete extraction of all selected conditions.
- Consistent gene identifiers.
- No NaN or infinite activation values.

The validation results were recorded in:

`results/extraction_validation/full_hdf5_validation.json`

An extraction manifest was also generated:

`results/extraction_validation/extraction_manifest.json`

The manifest records the PIE source revision, checkpoint SHA-256 checksum, dataset sampling parameters, selected condition indices, and references to the gene vocabulary and validation report.

Training and validation condition indices were reproduced exactly using the recorded sampling seeds.

### 8.5. Methodological Differences from the Reference Study

The extraction methodology follows the general principles of Kendiukhov's SAE research but is adapted to PIE.

Unlike Geneformer and scGPT, PIE uses an encoder–processor–decoder architecture with distinct latent and gene-specific representation spaces.

The implementation uses HDF5 instead of the NumPy memory-mapped storage employed in the reference pipeline.

The extracted observations also differ biologically: PIE representations are associated with perturbation conditions, latent slots, and output genes rather than the cell-level observations used in the original study.

These differences must be considered when interpreting SAE features and comparing them across layers.

### 8.6. Limitations and Next Steps

The expanded dataset contains 128 training and 32 validation experimental conditions. Multiple latent or gene-specific vectors derived from the same condition are not independent biological observations.

The full HDF5 validation establishes structural consistency and numerical finiteness but does not independently recompute every activation from the original PIE model.

The extracted datasets provide a reproducible foundation for layer-wise SAE training, feature characterization, cross-layer analysis, and biological interpretation.

The next stage will audit and adapt the original bio-sae training implementation, with particular attention to initialization, optimization, and experiment reproducibility.